<a href="https://colab.research.google.com/github/PinkCongolala/Automation-Codes/blob/main/Metrics_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Working Code for Auto Metrics Extraction

PROTOTYPE DOWNLOADER WITH BUILT IN QUERY ENCHANCEMENT. USE THE NEW ONE BELOW THIS CODE

In [ ]:
!pip install smartsheet-python-sdk pandas openpyxl

import os
import shutil
import zipfile
import smartsheet
import pandas as pd
from google.colab import files
from datetime import datetime
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- ACCESS TOKEN IN SMARTSHEET ---
ACCESS_TOKEN = "mHm7ovoYkHeuLmYanZPtfqbEBfpHVYMogp4A7"

client = smartsheet.Smartsheet(ACCESS_TOKEN)
client.errors_as_exceptions(True)

# --- CIS sheet mapping ---
cis_sheets = {
    "Apex": ["Metrics Apex 2026 Apr","Metrics Apex 2026 Apr W1","Metrics Apex 2026 Apr W2","Metrics Apex 2026 Apr W3","Metrics Apex 2026 Apr W4"],
    "Champions": ["Metrics Champions 2026 Apr","Metrics Champions 2026 Apr W1","Metrics Champions 2026 Apr W2","Metrics Champions 2026 Apr W3","Metrics Champions 2026 Apr W4"],
    "Dominators": ["Metrics Dominators 2026 Apr","Metrics Dominators 2026 Apr W1","Metrics Dominators 2026 Apr W2","Metrics Dominators 2026 Apr W2.1","Metrics Dominators 2026 Apr W3","Metrics Dominators 2026 Apr W3.1","Metrics Dominators 2026 Apr W4"],
    "Stratos": ["Metrics Stratos 2026 Apr","Metrics Stratos 2026 Apr W1","Metrics Stratos 2026 Apr W2","Metrics Stratos 2026 Apr W3","Metrics Stratos 2026 Apr W4"],
    "Kingsmen": ["Metrics Kingsmen 2026 Apr"],
    "The Luminaries": ["Metrics The Luminaries 2026 Apr","Metrics The Luminaries 2026 Apr W1","Metrics The Luminaries 2026 Apr W2","Metrics The Luminaries 2026 Apr W3","Metrics The Luminaries 2026 Apr W4"],
    "The Royals": ["Metrics The Royals 2026 Apr","Metrics The Royals 2026 Apr W1","Metrics The Royals 2026 Apr W2","Metrics The Royals 2026 Apr W3","Metrics The Royals 2026 Apr W4"],
    "The Elite": ["Metrics The Elite 2026 Apr","Metrics The Elite 2026 Apr W1","Metrics The Elite 2026 Apr W2","Metrics The Elite 2026 Apr W3","Metrics The Elite 2026 Apr W4"],
    "Titans": ["Metrics Titans 2026 Apr"],
    "One Legacy": ["Metrics One Legacy 2026 Apr"]
}

available_divisions = list(cis_sheets.keys())

# --- FAST LOOKUP ---
all_sheets = client.Sheets.list_sheets(include_all=True).data
sheet_lookup = {s.name.lower().strip(): s.id for s in all_sheets}

# --- USER INPUT ---
choice = input("Download all or one division? (all/one): ").strip().lower()
selected_divisions = available_divisions.copy()

if choice == "one":
    for i, d in enumerate(available_divisions, 1):
        print(f"{i}. {d}")

    sel = input("Pick division (name or number): ").strip()

    if sel.isdigit():
        selected_divisions = [available_divisions[int(sel)-1]]
    else:
        matches = [d for d in available_divisions if sel.lower() in d.lower()]
        if len(matches) == 1:
            selected_divisions = matches
        else:
            print("Invalid or multiple matches. Defaulting to ALL.")

# --- EXPORT FOLDER ---
base_dir = "CIS_exports"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir)

# --- DOWNLOAD ---
for div in selected_divisions:
    for sheet_name in cis_sheets.get(div, []):
        key = sheet_name.lower().strip()
        sheet_id = sheet_lookup.get(key)

        print(f"🔍 {div} - {sheet_name}")

        if not sheet_id:
            print(f"⚠️ Not found: {sheet_name}")
            continue

        client.Sheets.get_sheet_as_excel(
            sheet_id,
            base_dir,
            alternate_file_name=f"CIS {sheet_name}.xlsx"
        )

        print(f"✅ Exported {sheet_name}")

# --- CONSOLIDATION ---
print("\n📊 Consolidating files...")

all_data = []

for file in os.listdir(base_dir):
    if file.endswith(".xlsx"):
        path = os.path.join(base_dir, file)

        try:
            df = pd.read_excel(path)
            df["Source_File"] = file
            df["Division"] = file.split(" ")[1] if " " in file else "Unknown"
            all_data.append(df)

            print(f"✅ Loaded {file}")

        except Exception as e:
            print(f"⚠️ Failed: {file} | {e}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    consolidated_path = os.path.join(base_dir, "CIS_Consolidated.xlsx")
    final_df.to_excel(consolidated_path, index=False)
    print("\n🎉 Consolidated file created")
else:
    print("❌ No data to consolidate")

# --- ZIP ---
today = datetime.now().strftime("%Y_%b_%d").lower()

if len(selected_divisions) == 1:
    name = selected_divisions[0].replace(" ", "_")
    zip_name = f"{name}_CIS_{today}.zip"
else:
    zip_name = f"CIS_{today}_Sheets.zip"

with zipfile.ZipFile(zip_name, "w") as z:
    for root, _, files_list in os.walk(base_dir):
        for f in files_list:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, base_dir))

# --- CLEANUP ---
shutil.rmtree(base_dir)

print(f"\n🎉 Done: {zip_name}")
files.download(zip_name)

(NEW) OPTIMIZED DOWNLOADER WITH CHOICE

In [ ]:
!pip install smartsheet-python-sdk

import os
import shutil
import zipfile
import smartsheet
from google.colab import files
from datetime import datetime
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- ACCESS TOKEN (HARDCODED AS REQUESTED) ---
ACCESS_TOKEN = "mHm7ovoYkHeuLmYanZPtfqbEBfpHVYMogp4A7"

client = smartsheet.Smartsheet(ACCESS_TOKEN)
client.errors_as_exceptions(True)

# --- CIS sheet mapping ---
cis_sheets = {
    "Apex": ["Metrics Apex 2026 Apr","Metrics Apex 2026 Apr W1","Metrics Apex 2026 Apr W2","Metrics Apex 2026 Apr W3","Metrics Apex 2026 Apr W4"],
    "Champions": ["Metrics Champions 2026 Apr","Metrics Champions 2026 Apr W1","Metrics Champions 2026 Apr W2","Metrics Champions 2026 Apr W3","Metrics Champions 2026 Apr W4"],
    "Dominators": ["Metrics Dominators 2026 Apr","Metrics Dominators 2026 Apr W1","Metrics Dominators 2026 Apr W2","Metrics Dominators 2026 Apr W2.1","Metrics Dominators 2026 Apr W3","Metrics Dominators 2026 Apr W3.1","Metrics Dominators 2026 Apr W4"],
    "Stratos": ["Metrics Stratos 2026 Apr","Metrics Stratos 2026 Apr W1","Metrics Stratos 2026 Apr W2","Metrics Stratos 2026 Apr W3","Metrics Stratos 2026 Apr W4"],
    "Kingsmen": ["Metrics Kingsmen 2026 Apr"],
    "The Luminaries": ["Metrics The Luminaries 2026 Apr","Metrics The Luminaries 2026 Apr W1","Metrics The Luminaries 2026 Apr W2","Metrics The Luminaries 2026 Apr W3","Metrics The Luminaries 2026 Apr W4"],
    "The Royals": ["Metrics The Royals 2026 Apr","Metrics The Royals 2026 Apr W1","Metrics The Royals 2026 Apr W2","Metrics The Royals 2026 Apr W3","Metrics The Royals 2026 Apr W4"],
    "The Elite": ["Metrics The Elite 2026 Apr","Metrics The Elite 2026 Apr W1","Metrics The Elite 2026 Apr W2","Metrics The Elite 2026 Apr W3","Metrics The Elite 2026 Apr W4"],
    "Titans": ["Metrics Titans 2026 Apr"],
    "One Legacy": ["Metrics One Legacy 2026 Apr"]
}

available_divisions = list(cis_sheets.keys())

# --- FAST LOOKUP (major optimization) ---
all_sheets = client.Sheets.list_sheets(include_all=True).data
sheet_lookup = {s.name.lower().strip(): s.id for s in all_sheets}

# --- USER INPUT ---
choice = input("Download all or one division? (all/one): ").strip().lower()
selected_divisions = available_divisions.copy()

if choice == "one":
    for i, d in enumerate(available_divisions, 1):
        print(f"{i}. {d}")

    sel = input("Pick division (name or number): ").strip()

    if sel.isdigit():
        selected_divisions = [available_divisions[int(sel)-1]]
    else:
        matches = [d for d in available_divisions if sel.lower() in d.lower()]
        if len(matches) == 1:
            selected_divisions = matches
        else:
            print("Invalid or multiple matches. Defaulting to ALL.")

# --- EXPORT FOLDER ---
base_dir = "CIS_exports"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir)

# --- EXPORT ---
for div in selected_divisions:
    for sheet_name in cis_sheets.get(div, []):
        key = sheet_name.lower().strip()
        sheet_id = sheet_lookup.get(key)

        print(f"🔍 {div} - {sheet_name}")

        if not sheet_id:
            print(f"⚠️ Not found: {sheet_name}")
            continue

        client.Sheets.get_sheet_as_excel(
            sheet_id,
            base_dir,
            alternate_file_name=f"CIS {sheet_name}.xlsx"
        )

        print(f"✅ Exported {sheet_name}")

# --- ZIP ---
today = datetime.now().strftime("%Y_%b_%d").lower()

if len(selected_divisions) == 1:
    name = selected_divisions[0].replace(" ", "_")
    zip_name = f"{name}_CIS_{today}.zip"
else:
    zip_name = f"CIS_{today}_Sheets.zip"

with zipfile.ZipFile(zip_name, "w") as z:
    for root, _, files_list in os.walk(base_dir):
        for f in files_list:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, base_dir))

# --- CLEANUP ---
shutil.rmtree(base_dir)

print(f"\n🎉 Done: {zip_name}")
files.download(zip_name)

(OLD) Prototype downloader with choice






In [ ]:
!pip install smartsheet-python-sdk # PARA GUMANA YUNG CODE INSTALL THIS

import os
import shutil
import zipfile
import smartsheet
from google.colab import files
import warnings
from datetime import datetime

warnings.filterwarnings("ignore", category=DeprecationWarning)

ACCESS_TOKEN = "mHm7ovoYkHeuLmYanZPtfqbEBfpHVYMogp4A7"
client = smartsheet.Smartsheet(ACCESS_TOKEN)
client.errors_as_exceptions(True)

# --- CIS sheet mapping (Dominators included) ---
cis_sheets = {
    "Apex": ["Metrics Apex 2026 Apr","Metrics Apex 2026 Apr W1", "Metrics Apex 2026 Apr W2","Metrics Apex 2026 Apr W3","Metrics Apex 2026 Apr W4"  ],
    "Champions": ["Metrics Champions 2026 Apr","Metrics Champions 2026 Apr W1","Metrics Champions 2026 Apr W2","Metrics Champions 2026 Apr W3","Metrics Champions 2026 Apr W4" ]  ,
    "Dominators": ["Metrics Dominators 2026 Apr","Metrics Dominators 2026 Apr W1","Metrics Dominators 2026 Apr W2","Metrics Dominators 2026 Apr W2.1",  "Metrics Dominators 2026 Apr W3",  "Metrics Dominators 2026 Apr W3.1", "Metrics Dominators 2026 Apr W4"],
    "Stratos": ["Metrics Stratos 2026 Apr","Metrics Stratos 2026 Apr W1","Metrics Stratos 2026 Apr W2","Metrics Stratos 2026 Apr W3","Metrics Stratos 2026 Apr W4" ],
    "Kingsmen": "Metrics Kingsmen 2026 Apr",
    "The Luminaries": ["Metrics The Luminaries 2026 Apr","Metrics The Luminaries 2026 Apr W1","Metrics The Luminaries 2026 Apr W2","Metrics The Luminaries 2026 Apr W3","Metrics The Luminaries 2026 Apr W4"],
    "The Royals": ["Metrics The Royals 2026 Apr", "Metrics The Royals 2026 Apr W1", "Metrics The Royals 2026 Apr W2", "Metrics The Royals 2026 Apr W3","Metrics The Royals 2026 Apr W4" ],
    "The Elite": ["Metrics The Elite 2026 Apr", "Metrics The Elite 2026 Apr W1", "Metrics The Elite 2026 Apr W2","Metrics The Elite 2026 Apr W3","Metrics The Elite 2026 Apr W4" ],
    "Titans": "Metrics Titans 2026 Apr",
    "One Legacy": "Metrics One Legacy 2026 Apr"
}

# --- CMF sheet mapping ---
cmf_sheets = {
    "Apex": "CMF Apex 2026 Apr",
    "Champions": "CMF Champions 2026 Apr",
    "Dominators": ["CMF Dominators 2026 Apr", "CMF Dominators 2026 Apr W1","CMF Dominators 2026 Apr W2" ],
    "Stratos": "CMF Stratos 2026 Apr",
    "Kingsmen": "CMF Kingsmen 2026 Apr",
    "The Luminaries": "CMF The Luminaries 2026 Apr",
    "The Royals": "CMF The Royals 2026 Apr",
    "The Elite": "CMF The Elite 2026 Apr",
    "Titans": "CMF Titans 2026 Apr",
    "One Legacy": "CMF One Legacy 2026 Apr"
}

# --- Helper to get a clean mapping for case-insensitive lookups ---
available_divisions = list(cis_sheets.keys())
lower_to_div = {d.lower(): d for d in available_divisions}

def ask_download_type():
    while True:
        ans = input("Do you want to download CIS, CMF, or both? (cis/cmf/both): ").strip().lower()
        if ans in ("cis", "c"):
            return "cis"
        if ans in ("cmf", "m"):
            return "cmf"
        if ans in ("both", "b"):
            return "both"
        print("Please type 'cis', 'cmf', or 'both'.")

def ask_all_or_one():
    while True:
        ans = input("Do you want to download all divisions or only one? (all/one): ").strip().lower()
        if ans in ("all", "a"):
            return "all"
        if ans in ("one", "o"):
            return "one"
        print("Please type 'all' or 'one' (or 'a'/'o').")

download_type = ask_download_type()
choice = ask_all_or_one()

selected_divisions = available_divisions.copy()

if choice == "one":
    print("\nAvailable divisions:")
    for i, div in enumerate(available_divisions, 1):
        print(f"{i}. {div}")
    while True:
        selection = input("\nType the division name or number (e.g., Dominators or 3): ").strip()
        if selection.isdigit():
            idx = int(selection) - 1
            if 0 <= idx < len(available_divisions):
                selected_divisions = [available_divisions[idx]]
                break
            else:
                print("Number out of range. Try again.")
                continue
        sel_low = selection.lower()
        matched = lower_to_div.get(sel_low)
        if matched:
            selected_divisions = [matched]
            break
        else:
            matches = [d for d in available_divisions if sel_low in d.lower()]
            if len(matches) == 1:
                selected_divisions = [matches[0]]
                print(f"Auto-selected: {matches[0]}")
                break
            elif len(matches) > 1:
                print(f"Multiple matches found: {', '.join(matches)}. Please be more specific or use the number.")
            else:
                print("No matching division found. Please type exact name (case-insensitive) or number.")

# --- Metricsorary folder to store all sheets ---
Metrics_dir = "Metrics_exports"
if os.path.exists(Metrics_dir):
    shutil.rmtree(Metrics_dir)
os.makedirs(Metrics_dir, exist_ok=True)

all_sheets = client.Sheets.list_sheets(include_all=True).data

# --- Function to export sheets into a specific subfolder ---
def export_sheets(sheet_mapping, subfolder, prefix="", divisions=None):
    folder_path = os.path.join(Metrics_dir, subfolder)
    os.makedirs(folder_path, exist_ok=True)

    for div in divisions:
        if div not in sheet_mapping:
            print(f"⚠️ {div} not found in {subfolder} mapping. Skipping.")
            continue

        sheet_names = sheet_mapping[div]
        if not isinstance(sheet_names, list):
            sheet_names = [sheet_names]

        for sheet_name in sheet_names:
            print(f"🔍 Processing {div} - {sheet_name} {prefix}...")

            target = next(
                (s for s in all_sheets if sheet_name.lower().strip() == s.name.lower().strip()),
                None
            )

            if not target:
                print(f"⚠️ Sheet not found: {sheet_name}")
                continue

            client.Sheets.get_sheet_as_excel(
                target.id,
                folder_path,
                alternate_file_name=f"{prefix}{sheet_name}.xlsx"
            )
            print(f"✅ Exported {prefix}{sheet_name}")

# --- Export based on selected type ---
if download_type in ("cis", "both"):
    export_sheets(cis_sheets, subfolder="CIS", prefix="CIS ", divisions=selected_divisions)

if download_type in ("cmf", "both"):
    export_sheets(cmf_sheets, subfolder="CMF", prefix="", divisions=selected_divisions)

# --- Create final zip with subfolders ---
today = datetime.now().strftime("%Y_%b_%d").lower()

if len(selected_divisions) == 1:
    div_name = selected_divisions[0].replace(" ", "_")
    if download_type == "cis":
        final_zip = f"{div_name}_CIS_{today}.zip"
    elif download_type == "cmf":
        final_zip = f"{div_name}_CMF_{today}.zip"
    else:
        final_zip = f"{div_name}_CIS_and_CMF_{today}.zip"
else:
    if download_type == "cis":
        final_zip = f"CIS_{today}_Sheets.zip"
    elif download_type == "cmcisf":
        final_zip = f"CMF_{today}_Sheets.zip"
    else:
        final_zip = f"CIS_and_CMF_{today}_Sheets.zip"

with zipfile.ZipFile(final_zip, "w") as zipf:
    for root, _, files_list in os.walk(Metrics_dir):
        for file in files_list:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, Metrics_dir)
            zipf.write(full_path, arcname)

# --- Cleanup Metrics folder ---
shutil.rmtree(Metrics_dir)

print(f"\n🎉 All done! Files zipped into {final_zip}")
files.download(final_zip)
